# Independent final V037 audit before submission

Prepared while the final fit is running, before its predictions or official score exist. This notebook does no fitting or parameter selection. It rebuilds approved training features and normalization, verifies the fixed 190-epoch all-data recipe and five accepted reused models, reconstructs all 603,600 outputs independently, and checks exact CSV alignment and round-trip values.

The independent CPU cache check uses 64 test breaths chosen with seed **20263701**, fixed before final outputs. Its tolerance is one hundredth of the median spacing of the approved all-training inspiratory pressure grid. Production loader guards remain unchanged. Final fitting has no independent holdout; earlier grouped confirmation supplies the evidence of improvement.

Run only after the verified final archive has been safely published: set `PA3_RUN_FINAL_AUDIT_V037=1`. A successful immutable receipt is written to `research/final_review_v037.json`. Official mean MAE is assessed only after a valid Gym submission.


In [1]:
from pathlib import Path
from datetime import datetime,timezone
import os,json,hashlib,math,gc,time,copy
import numpy as np
import pandas as pd
import torch

FINAL_SOURCE='68a4aa07834bd8cd365058631f01279620117c7a896657c9a3029622e3d4ed64'
F_RESULT='ed5c3f254a563cc6503bece1834d5ed8c4f1fc6a51f52b3f9d9f158aacfdfcbf'
F_AUDIT='3a757c33687785acba1858c41302f57d3118c79480db7aff72f18f566df69a59'
APPROVED={'train.csv':'2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7','test.csv':'1ed86b32480e93397cc961dfd09af2d0bedfbab901921eac44de8ed78c3f1672','sample_submission.csv':'de4c137a22804654b597723f32470b24f23d9ef662205eca0fe31b68970f0594'}
ACCEPTED_SHA='3746753aa6cd7eaa306f2279e24d5a5231b6bae9c700bcd703e54b756a823aaf'
BEST_CSV='274668c4dc14e9d20a554c738859fca37e448836797d7260ec7b11464dcde3e2'
WEIGHTS=dict(original_compact=.125,deep=.25,curve=.125,memory=.125,offset=.125,capacity_laplace_curve=.25)
CPU_SAMPLE_SEED=20263701
CPU_SAMPLE_BREATHS=64
checks={}
def require(name,value):
    if not bool(value):raise AssertionError(name)
    checks[name]=True

def read(path):return json.loads(Path(path).read_text())
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def code(path):return '\n'.join(''.join(c['source']) for c in read(path)['cells'] if c['cell_type']=='code')
def code_sha(path):return hashlib.sha256(code(path).encode()).hexdigest()
def rawhash(a):return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
def arrayhash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256(str((a.shape,a.dtype.str)).encode());h.update(memoryview(a).cast('B'));return h.hexdigest()
def digest(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def safe(v):
    if isinstance(v,dict):return {k:safe(x) for k,x in v.items()}
    if isinstance(v,(list,tuple)):return [safe(x) for x in v]
    if isinstance(v,np.ndarray):return v.tolist()
    if isinstance(v,np.generic):return v.item()
    return v

def local(root,value):
    value=str(value);marker='/ventilator-pressure-prediction/'
    p=(root/(value.split(marker,1)[1] if marker in value else value)).resolve()
    require('project-contained:'+str(p),p.is_relative_to(root));return p

def components(root):
    require('Frozen final notebook source',code_sha(root/'final_fit_v037.ipynb')==FINAL_SOURCE)
    # Load definitions only. No model fitter, queue, optimizer or run entry invoked.
    for key in list(os.environ):
        if key.startswith('PA3_RUN_'):os.environ[key]='0'
    ns={'__name__':'v037_independent_final_audit'}
    exec(compile(code(root/'final_fit_v037.ipynb'),'final_fit_v037.ipynb','exec'),ns)
    for name,h in ns['V037_SOURCE_DEPENDENCIES'].items():
        require('Frozen dependency:'+name,code_sha(root/name)==h)
        exec(compile(code(root/name),name,'exec'),ns)
    oldlock=read(root/'outputs/confirmation_v011/locked_before_D.json')
    ns['v011_load_components'](root,oldlock['source_fingerprints'])
    require('Frozen Laplace component',code_sha(root/'research/laplace_components_v029.ipynb')==ns['V037_LAPLACE_SHA'])
    exec(compile(code(root/'research/laplace_components_v029.ipynb'),'laplace_components_v029.ipynb','exec'),ns)
    torch.set_num_threads(4)
    return ns,oldlock

def independent_policy(predictions,weights,grid,phase):
    # Integer replication + ordinary median is independent of production's sort/cumulative-weight implementation.
    names=list(weights);multiplicity=np.array([round(8*weights[n]) for n in names],dtype=np.int64)
    require('Exact eighth-unit weights',sum(multiplicity)==8 and all(multiplicity[i]/8==weights[n] for i,n in enumerate(names)))
    values=np.stack([predictions[n] for n in names],axis=1)
    med=np.median(np.repeat(values,multiplicity,axis=1),axis=1)
    hi=np.clip(np.searchsorted(grid,med),0,len(grid)-1);lo=np.maximum(hi-1,0)
    snapped=np.where(abs(med-grid[lo])<=abs(med-grid[hi]),grid[lo],grid[hi])
    mean=sum(weights[n]*predictions[n] for n in names)
    return np.where(phase==0,snapped,mean)

def audit_final_v037(root='.'):
    root=Path(root).resolve();out=root/'outputs/confirmation_v036/final_all_v037';began=time.monotonic();checks.clear()
    require('Completed final artifacts exist',(out/'final_manifest.json').is_file())
    m=read(out/'final_manifest.json');design=read(out/'design.json');plan=read(root/'research/confirmation_protocol_v036.json')
    require('Complete native final stage',m['status']=='complete' and m['submitted'] is False)
    require('Final design hash',digest(design)==m['design_sha256'])
    require('Only frozen selected ensemble',m['weights']==design['weights']==plan['winner_weights']==WEIGHTS and set(m['members'])==set(m['prediction_members'])==set(WEIGHTS))
    require('Full approved training/no independent holdout',m['all_training_breaths']==design['all_training_breaths']==67905 and m['scored_training_points']==2061922 and m['final_fit_has_independent_holdout'] is False and design['final_fit_has_independent_holdout'] is False)
    require('Fixed output policies',m['aggregation']==design['aggregation']=='fixed weighted median midpoint then nearest all-training inspiratory grid' and m['unscored_phase_rule']==design['unscored_phase_rule']=='fixed six-member mean')
    require('Exact reviewed F result',sha(root/'outputs/confirmation_v036/confirmation_result.json')==F_RESULT)
    require('Exact independent complete F audit',sha(root/'research/confirmation_review_v036.json')==F_AUDIT)
    fr=read(root/'outputs/confirmation_v036/confirmation_result.json');fa=read(root/'research/confirmation_review_v036.json')
    require('Confirmed scientific pass',fr['status']=='complete' and fr['passed'] and fa['status']=='passed' and fa['passed'] and fa['confirmation_result_sha256']==F_RESULT and fr['gain']>0 and fr['interval']['high']<0)
    ns,oldlock=components(root)
    gate_plan,gate=ns['v037_confirmation_gate'](root,F_RESULT,F_AUDIT);gate['final_source_code_sha256']=FINAL_SOURCE
    require('Exact unchanged completed F gate',gate==m['gate']==design['gate'] and gate_plan==plan)
    accepted_path=root/'outputs/confirmation_v011/final_all/final_manifest.json';accepted=read(accepted_path)
    require('Accepted all-data provenance preserved',sha(accepted_path)==ACCEPTED_SHA==design['accepted_manifest_sha256'])
    for rel,h in [('outputs/confirmation_v014/final_all_v018/submission.csv',BEST_CSV),('outputs/confirmation_v011/final_all/submission.csv','c344a485f5ef1290aadf5dfd379333009efab674b7ccdff6f361e225a4b8b77a'),('outputs/improvements_v005/confirmation_round1/final_all/submission.csv','ff149f3ab87e1e52b948f536c9fdf9a526471ac0723b6b785bf84c053ebed607')]:require('Preserved accepted CSV:'+rel,sha(root/rel)==h)
    require('Old accepted CSV recorded',m['old_CSV_sha256']==design['accepted_CSV_sha256']==accepted['submission_sha256'])
    for name,h in APPROVED.items():require('Approved data:'+name,sha(root/'data'/name)==h)
    for name,h in m['source_fingerprints'].items():
        if name.endswith('.ipynb'):
            p=root/name if (root/name).exists() else root/'research'/name
            require('Recorded notebook source:'+name,code_sha(p)==h)
    require('New source receipts bound',m['source_fingerprints']['confirmed_F_result']==F_RESULT and m['source_fingerprints']['confirmed_F_audit']==F_AUDIT and m['source_fingerprints']['final_fit_v037.ipynb']==FINAL_SOURCE and design['source_fingerprints']==m['source_fingerprints'])
    proof=read(out/'course_proofs.json')
    require('Actual CUDA final-consumer proof',proof['status']=='passed' and proof['actual_cuda'] and proof['check_count']==88 and all(proof['checks'].values()) and proof['driver_source_code_sha256']==FINAL_SOURCE and proof['F_result_sha256']==F_RESULT and proof['F_audit_sha256']==F_AUDIT)
    # Rebuild every approved training row in canonical CSV order; statistics use no test data.
    frame=pd.read_csv(root/'data/train.csv',dtype=dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_in='float32',u_out='int8',pressure='float32'))
    y=frame.pressure.to_numpy(copy=True).reshape(-1,80);phase=frame.u_out.to_numpy(copy=True).reshape(-1,80);bids=frame.breath_id.to_numpy().reshape(-1,80)[:,0];rowids=frame.id.to_numpy()
    require('All approved groups/scored rows',len(y)==67905 and int((phase==0).sum())==2061922 and len(np.unique(bids))==67905)
    require('Canonical all-data group/row hashes',rawhash(bids)==m['training']['group_sha256']==accepted['all_training_group_sha256'] and rawhash(rowids)==accepted['all_training_row_ids_sha256'])
    grid=np.unique(y[phase==0]).astype(np.float64);known=frame.drop(columns='pressure');del frame;gc.collect()
    x,names,xbids=ns['make_sequence_features'](known,'combined');del known;gc.collect()
    require('All approved 47-feature tensors',x.shape==(67905,80,47) and np.array_equal(xbids,bids) and names==ns['feature_names_v005']('combined'))
    fingerprints=dict(x_sha256=arrayhash(x),y_sha256=arrayhash(y),out_sha256=arrayhash(phase))
    require('Independently rebuilt training fingerprints',fingerprints==m['training']['data_fingerprints']==accepted['all_training_data_fingerprints'])
    prep=ns['fit_preprocessor'](x,y,phase,plan['members']['capacity_laplace_curve']['model_spec'],names,scaler_phase='all',residual_baseline=False)
    require('All-training-only normalization',ns['_v8_statistics_sha256'](prep)==m['training']['preprocessing_sha256']==accepted['all_training_preprocessing_sha256'])
    del x,y,phase,rowids;gc.collect()
    test=pd.read_csv(root/'data/test.csv',dtype={k:v for k,v in ns['V013_DTYPES'].items() if k!='pressure'});ids=test.id.to_numpy(copy=True);testphase=test.u_out.to_numpy(copy=True)
    sample=pd.read_csv(root/'data/sample_submission.csv',usecols=['id'],dtype=dict(id='int64')).id.to_numpy()
    require('603600 unique canonical test/sample IDs',len(ids)==603600 and len(np.unique(ids))==603600 and np.array_equal(ids,sample) and np.array_equal(ids,np.load(out/'test_row_ids.npy',allow_pickle=False)))
    tx,tnames,tbids=ns['make_sequence_features'](test,'combined');del test;gc.collect()
    require('Approved test feature schema',tx.shape==(7545,80,47) and tnames==names)
    selected=np.sort(np.random.default_rng(CPU_SAMPLE_SEED).choice(7545,CPU_SAMPLE_BREATHS,replace=False));sample_raw=tx[selected].copy();del tx;gc.collect()
    summaries={};predictions={};cache_differences={};spacing=float(np.median(np.diff(grid)));require('Positive native grid spacing',spacing>0 and np.isfinite(spacing))
    for name in WEIGHTS:
        r=m['members'][name];recipe=plan['members'][name];budget=recipe['selected_epochs'];cp=local(root,r['checkpoint_path']);rp=local(root,r['prior_report_file'] if name in ['deep','original_compact'] else r['report_file']);rh=r['prior_report_sha256'] if name in ['deep','original_compact'] else r['report_sha256'];report=read(rp)
        require(name+':checkpoint/report raw hashes',sha(cp)==r['checkpoint_sha256'] and sha(rp)==rh)
        require(name+':frozen recipe',r['recipe']==recipe and recipe['seed']==20261102)
        ck=torch.load(cp,map_location='cpu',weights_only=False);steps=math.ceil(67905/recipe['train_cfg']['batch_size'])
        require(name+':fixed complete optimizer budget',ck['completed'] and ck['termination']=='fixed_A_budget' and not ck['validation_used'] and ck['epoch']==len(ck['epochs'])==budget and ck['optimizer_steps']==steps*budget)
        require(name+':exact model seed/config',ck['seed']==recipe['seed'] and ck['model_spec']==ns['canonical_model_spec'](recipe['model_spec']) and ck['config']==ns['_train_configuration'](recipe['train_cfg']))
        require(name+':exact selected LR schedule',ck['selected_lr_schedule']==[e['lr_used'] for e in ck['epochs']]==recipe['lr_schedule']==recipe['train_cfg']['frozen_lr_schedule'][:budget])
        require(name+':independent all-data preprocessing',safe(ck['preprocess'])==safe(prep))
        fp=ck['fingerprint'];require(name+':all-data fixed fingerprints',fp['mode']=='fixed' and all(fp[k]==v for k,v in fingerprints.items()) and fp['feature_names']==names and fp['extra']==dict(lr_schedule=recipe['lr_schedule']))
        for i,e in enumerate(ck['epochs'],1):
            require(name+':epoch'+str(i),e['epoch']==i and e['optimizer_steps']==steps and e['cumulative_optimizer_steps']==i*steps and e['training_scored_points']==2061922 and not e['validation_used'] and 'validation_inspiratory_mae' not in e)
        require(name+':complete report',report['completed'] and not report['validation_used'] and report['training_breaths']==67905 and report['executed_epochs']==budget and report['optimizer_steps']==steps*budget and report['selected_lr_schedule']==report['executed_lr_schedule']==recipe['lr_schedule'])
        require(name+':finite learned tensors',all(bool(torch.isfinite(v).all()) for v in ck['state_dict'].values()))
        if name=='capacity_laplace_curve':
            require(name+':exact fixed new budget',budget==190 and steps==266 and ck['optimizer_steps']==50540 and r['training_breaths']==67905 and r['selected_epochs']==190)
            require(name+':new identity after confirmed gate',r['identity']==read(out/name/'identity.json') and r['identity']['gate']==gate and r['identity']['recipe']==recipe and r['identity']['sources']==m['source_fingerprints'] and r['identity']['training_group_sha256']==rawhash(bids) and r['identity']['training_data_fingerprints']==fingerprints)
            require(name+':new sources and actual objective',r['source_fingerprints']==m['source_fingerprints']==ck['v008']['source_fingerprints'] and ck['v029']['objective']==ns['V029_OBJECTIVE'] and ck['v029']['arm']==r['arm']=='v037_all_capacity_laplace_curve' and ck['actual_training_objective']==report['actual_training_objective']=='laplace_curve' and ck['v008']['loss_kind']=='curve_l1')
            lastpath=local(root,r['last_checkpoint_path']);require(name+':last checkpoint hash',sha(lastpath)==r['last_checkpoint_sha256']);last=torch.load(lastpath,map_location='cpu',weights_only=False)
            require(name+':resumable final state',last['completed'] and last['epoch']==budget and last['optimizer_steps']==50540 and all(torch.equal(v,last['state_dict'][k]) for k,v in ck['state_dict'].items()) and 'optimizer_state' in last)
            del last
            model=ns['v037_laplace_factory'](recipe['model_spec'],47,ck['preprocess'])
        else:
            old=accepted['members'][name]
            require(name+':accepted all-data reuse only',r['reuse_controller']=='v037' and r['accepted_manifest_sha256']==ACCEPTED_SHA and r['checkpoint_sha256']==old['checkpoint_sha256'] and r['recipe']==old['recipe'] and r['training_breaths']==67905 and r['scored_training_points']==2061922 and r['group_sha256']==rawhash(bids))
            require(name+':old record untouched',{k:v for k,v in r.items() if k not in ['reuse_controller','accepted_manifest_sha256']}==old)
            if name in ['deep','original_compact']:
                require(name+':prior accepted record hash',sha(local(root,r['prior_record_file']))==r['prior_record_sha256'])
                model=ns['V5PressureRNN'](recipe['model_spec'],47)
            else:
                require(name+':accepted wrapped source/objective',ck['v008']['source_fingerprints']==accepted['source_fingerprints'] and ck['v008']['arm']==r['arm'] and ck['v008']['loss_kind']==recipe['loss_kind'])
                model=ns['v011_factory'](name)(recipe['model_spec'],47,ck['preprocess'])
        require(name+':architecture shapes and parameter count',{k:tuple(v.shape) for k,v in model.state_dict().items()}=={k:tuple(v.shape) for k,v in ck['state_dict'].items()} and sum(v.numel() for v in model.parameters())==report['parameter_count'])
        if name=='capacity_laplace_curve':require(name+':capacity5108162',report['parameter_count']==5108162)
        meta=m['prediction_members'][name];path=local(root,meta['file']);p=np.load(path,allow_pickle=False)
        require(name+':native cache hash/shape/identity',sha(path)==meta['file_sha256'] and rawhash(p)==meta['array_sha256'] and p.dtype==np.float64 and p.shape==(603600,) and np.isfinite(p).all() and meta['row_ids_sha256']==rawhash(ids) and meta['test_sha256']==APPROVED['test.csv'] and meta['checkpoint_sha256']==r['checkpoint_sha256'])
        if name!='capacity_laplace_curve':require(name+':accepted native cache unchanged',meta==accepted['prediction_members'][name])
        else:require(name+':new cache sources',meta['identity']['sources']==m['source_fingerprints'] and meta['identity']['arm']=='v037_all_capacity_laplace_curve')
        # Manual independent CPU reconstruction, after exact frozen source, state shape and data checks.
        # Production's Python-version-specific bytecode guards are left unchanged.
        model.load_state_dict(ck['state_dict'],strict=True);model.eval();transformed=ns['transform_inputs'](sample_raw,ck['preprocess']);native=ns['_predict_native'](model,transformed,ck['preprocess'],torch.device('cpu'),32)
        diff=float(np.max(abs(native.astype(np.float64)-p.reshape(7545,80)[selected])))
        require(name+':64-breath independent CPU inference',diff<spacing/100);cache_differences[name]=diff
        predictions[name]=p;summaries[name]=dict(checkpoint_sha256=r['checkpoint_sha256'],epochs=budget,optimizer_steps=ck['optimizer_steps'],seed=ck['seed'],parameter_count=report['parameter_count'],training_seconds=report['elapsed_seconds'],accepted_reuse=name!='capacity_laplace_curve')
        del ck,model,transformed,native;gc.collect()
    require('Full all-training pressure grid',np.array_equal(grid,np.load(local(root,m['grid_file']),allow_pickle=False)) and sha(local(root,m['grid_file']))==m['grid_file_sha256'] and len(grid)==m['grid_levels'])
    expected=independent_policy(predictions,m['weights'],grid,testphase);saved=np.load(out/'final_test_predictions.npy',allow_pickle=False)
    require('Independent whole-row frozen native policy',np.array_equal(expected,saved) and saved.dtype==np.float64 and saved.shape==(603600,) and np.isfinite(saved).all())
    csvpath=local(root,m['submission']['file']);csv=pd.read_csv(csvpath,dtype=dict(id='int64',pressure='float64'),float_precision='round_trip')
    require('Exact CSV columns, canonical IDs and native values',list(csv.columns)==['id','pressure'] and len(csv)==603600 and np.array_equal(ids,csv.id.to_numpy()) and np.array_equal(expected,csv.pressure.to_numpy()) and np.isfinite(csv.pressure.to_numpy()).all())
    require('Final hashes agree',sha(csvpath)==m['submission_sha256']==m['submission']['file_sha256'] and sha(out/'final_test_predictions.npy')==m['prediction_file_sha256'] and rawhash(expected)==m['prediction_array_sha256']==m['submission']['native_prediction_array_sha256'] and rawhash(ids)==m['row_ids_array_sha256']==m['submission']['row_ids_array_sha256'])
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),check_count=len(checks),checks=checks,submission_file=str(csvpath.relative_to(root)),submission_sha256=sha(csvpath),final_manifest_sha256=sha(out/'final_manifest.json'),audit_notebook='audit_final_v037.ipynb',audit_code_sha256=code_sha(root/'audit_final_v037.ipynb'),final_source_code_sha256=FINAL_SOURCE,F_result_sha256=F_RESULT,F_audit_sha256=F_AUDIT,members=summaries,CPU_sample_seed=CPU_SAMPLE_SEED,CPU_sample_breaths=CPU_SAMPLE_BREATHS,CPU_cache_maximum_differences=cache_differences,CPU_cache_difference_limit=spacing/100,all_rows=603600,all_approved_training_breaths=67905,scored_training_points=2061922,final_fit_has_independent_holdout=False,official_best_preserved=.17427,official_target=.1638,no_training_or_new_selection=True,elapsed_seconds=time.monotonic()-began)
    with (root/'research/final_review_v037.json').open('x') as f:f.write(json.dumps(result,indent=2,allow_nan=False)+'\n')
    print(json.dumps({k:v for k,v in result.items() if k!='checks'},indent=2));return result


In [2]:
if os.environ.get('PA3_RUN_FINAL_AUDIT_V037','0')=='1':
    V037_FINAL_AUDIT=audit_final_v037(Path.cwd())


{
  "status": "passed",
  "created_utc": "2026-11-10T20:29:08.971666+00:00",
  "check_count": 1322,
  "submission_file": "outputs/confirmation_v036/final_all_v037/submission.csv",
  "submission_sha256": "a99e8acd59d2dbc6a2b50ce53ff6b108269afd8c010e6ba8504f27addc53f1e1",
  "final_manifest_sha256": "eaa7bf022bc88d292798ee0321aeb1ca00c1671b2730671dcda121735908da16",
  "audit_notebook": "audit_final_v037.ipynb",
  "audit_code_sha256": "cfe9c297d9e5aa627e7a33e590d941479781fd8af7e252614b7854ac579ff1a8",
  "final_source_code_sha256": "68a4aa07834bd8cd365058631f01279620117c7a896657c9a3029622e3d4ed64",
  "F_result_sha256": "ed5c3f254a563cc6503bece1834d5ed8c4f1fc6a51f52b3f9d9f158aacfdfcbf",
  "F_audit_sha256": "3a757c33687785acba1858c41302f57d3118c79480db7aff72f18f566df69a59",
  "members": {
    "original_compact": {
      "checkpoint_sha256": "19897b624599c57f2e4a7f976b2d2fdd4d0b5b1ba3f3dbf6884198ebdef23883",
      "epochs": 193,
      "optimizer_steps": 51338,
      "seed": 20261102,
      "pa

/Users/generalclasher/Documents/dsc102/pa3/ventilator-pressure-prediction/audit_final_v037.ipynb:1:170: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
